# LightDecoder smart cut: training the object model (U-2-Net)

This notebook fine-tunes the model LightDecoder uses for **Smart cut** on things that are not people
(`app/src/main/assets/models/u2net.onnx.0` + `.1`, the full U-2-Net, Apache-2.0) and writes a file the app
can use without any code change other than one number.

**Status: written but never run.** It was written where there is no GPU and no PyTorch, so only its
syntax was checked. Expect to fix small things on the first run (a download link, a folder name).

## The plan

| Step | What | Time on one T4 / RTX 3060-class GPU |
|---|---|---|
| 1 | Set up, fetch U-2-Net code and its published weights | 5 min |
| 2 | Fetch DUTS-TR (10,553 pictures with masks, what U-2-Net was trained on) | 5 min |
| 3 | Add **your own pictures**: the kinds of things you make stickers of | your time |
| 4 | Make clean masks for them with a large teacher model (BiRefNet), then **look at them and delete the bad ones** | ~1 s per picture + your time |
| 5 | Measure the published model on held-out pictures (the number to beat) | 2 min |
| 6 | Fine-tune: low learning rate, your pictures sampled more often than DUTS | 1-3 hours |
| 7 | Measure again. **Only go on if it is better on your pictures and not worse on DUTS** | 2 min |
| 8 | Export to ONNX in exactly the form the app expects, check it against PyTorch | 2 min |
| 9 | Split into two parts, copy into the app, change `FULL_OBJECTS_BYTES`, build | 10 min |

## What to expect, honestly

- The published weights are already trained on DUTS. Training on DUTS alone again gains nothing.
  **The gain comes from step 3 and 4**: a few hundred to a few thousand of your own kind of picture
  (statues, toys, pets, food, objects on cluttered tables) with clean masks.
- Step 4 is the trick that makes this practical: a model far too big for a phone (BiRefNet, needs a GPU)
  draws the masks, and the phone-sized model learns to copy it. That is how to get close to the iPhone's
  result on your pictures without drawing masks by hand.
- U-2-Net looks at the picture at 320x320. Fine hair and thin wires will stay softer than on an iPhone
  however long it trains; the app sharpens the edge afterwards from the full-size picture.
- 200 pictures of your own: a small but visible gain. 1,000-3,000: the worthwhile range.

## What the app expects (do not change these)

- Input `[1, 3, 320, 320]`, float32, RGB planes. The picture is squeezed to 320x320 (not cropped),
  each pixel divided by **the largest channel value in the picture**, then `(x - mean) / std` with
  mean `0.485, 0.456, 0.406` and std `0.229, 0.224, 0.225`. (`SubjectCutter.modelInput`)
- Output number 0: the mask, 320x320 values (`SubjectCutter.objectsWith`).
- The app accepts the file only if its size equals `FULL_OBJECTS_BYTES` in `SubjectCutter.kt`.

## 1. Set up

In [ ]:
import os, sys, subprocess, random, glob, shutil, zipfile, urllib.request
from pathlib import Path

WORK = Path("smartcut-work").resolve()      # everything this notebook makes goes here
WORK.mkdir(exist_ok=True)

def sh(*args, **kw):
    print("$", " ".join(map(str, args)))
    subprocess.run([str(a) for a in args], check=True, **kw)

# PyTorch itself should come from your notebook's image (Colab and Kaggle have it).
sh(sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "gdown", "pillow", "numpy", "tqdm")

import numpy as np, torch
from PIL import Image
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - training needs one")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(7); np.random.seed(7); torch.manual_seed(7)

In [ ]:
# U-2-Net's code (Apache-2.0) and its published weights (u2net.pth, 176 MB).
REPO = WORK / "U-2-Net"
if not REPO.exists():
    sh("git", "clone", "--depth", "1", "https://github.com/xuebinqin/U-2-Net", REPO)
sys.path.insert(0, str(REPO))
from model import U2NET

WEIGHTS = WORK / "u2net.pth"
if not WEIGHTS.exists():
    # The id is the one the U-2-Net README links for u2net.pth. If Google Drive refuses,
    # download u2net.pth by hand from the README's link and put it at WEIGHTS.
    import gdown
    gdown.download(id="1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ", output=str(WEIGHTS))
assert WEIGHTS.stat().st_size > 150_000_000, "u2net.pth did not download whole"

def published():
    net = U2NET(3, 1)
    net.load_state_dict(torch.load(WEIGHTS, map_location="cpu"))
    return net

## 2. DUTS-TR

10,553 pictures with masks. It keeps the model from forgetting what it already knows while it learns your pictures.
DUTS is published for research; the published U-2-Net weights were already trained on it.

In [ ]:
DUTS = WORK / "DUTS-TR"
if not DUTS.exists():
    z = WORK / "DUTS-TR.zip"
    if not z.exists():
        # Not reachable from where this was written, so unchecked. If it fails, get DUTS-TR.zip from the DUTS page by hand.
        urllib.request.urlretrieve("http://saliencydetection.net/duts/download/DUTS-TR.zip", z)
    with zipfile.ZipFile(z) as f:
        f.extractall(WORK)

def pairs(image_dir, mask_dir):
    """(picture, mask) for every picture that has a mask of the same name."""
    out = []
    for p in sorted(Path(image_dir).iterdir()):
        if p.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp"):
            continue
        m = Path(mask_dir) / (p.stem + ".png")
        if m.exists():
            out.append((p, m))
    return out

duts = pairs(DUTS / "DUTS-TR-Image", DUTS / "DUTS-TR-Mask")
print(len(duts), "DUTS pictures")   # expect 10553
assert len(duts) > 10000

## 3. Your own pictures

Put pictures in `smartcut-work/mine/images/`. Use the kinds of things you actually cut out, taken the way
you actually take them (phone camera, cluttered background, indoor light). Include the ones Smart cut gets
wrong today: those teach the most. One clear subject per picture.

Phone pictures are often stored sideways with a note saying how to turn them. The cell below turns them
upright and shrinks them so the rest of the notebook is fast.

In [ ]:
from PIL import ImageOps

MINE = WORK / "mine"
RAW, IMAGES, MASKS = MINE / "raw", MINE / "images", MINE / "masks"
for d in (RAW, IMAGES, MASKS):
    d.mkdir(parents=True, exist_ok=True)

# Drop originals in mine/raw (or straight into mine/images if they are already upright).
for p in sorted(RAW.iterdir()):
    out = IMAGES / (p.stem + ".jpg")
    if out.exists():
        continue
    try:
        im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
    except Exception as e:      # HEIC needs: pip install pillow-heif ; import pillow_heif ; pillow_heif.register_heif_opener()
        print("skipped", p.name, e); continue
    im.thumbnail((1600, 1600))
    im.save(out, quality=92)
print(len(list(IMAGES.iterdir())), "pictures of your own")

## 4. Masks for your pictures, drawn by a large teacher model

BiRefNet (MIT licence) is the model that was too big for the phone. Here it only draws the training masks.
It needs the GPU and about 6 GB of its memory.

**Then look at every mask.** The preview cell writes each picture next to its cut-out into
`mine/review/`. Delete from `mine/masks/` every mask that is wrong (or fix it in a paint program: white =
keep, black = remove). A wrong mask teaches the model the wrong thing; 300 checked pictures beat 3,000 unchecked.

If you already have masks (for example cut-outs exported from an iPhone as PNG with transparency), skip the
teacher: save the transparency as a white-on-black PNG of the same name in `mine/masks/`.

In [ ]:
sh(sys.executable, "-m", "pip", "install", "-q", "transformers", "timm", "kornia", "einops")
from transformers import AutoModelForImageSegmentation
import torchvision.transforms as T

teacher = AutoModelForImageSegmentation.from_pretrained("ZhengPeng7/BiRefNet", trust_remote_code=True).to(DEVICE).eval()
teacher_in = T.Compose([T.Resize((1024, 1024)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

todo = [p for p in sorted(IMAGES.iterdir()) if not (MASKS / (p.stem + ".png")).exists()]
for p in todo:
    im = Image.open(p).convert("RGB")
    with torch.no_grad():
        out = teacher(teacher_in(im).unsqueeze(0).to(DEVICE))[-1].sigmoid()[0, 0].float().cpu().numpy()
    Image.fromarray((out * 255).astype(np.uint8)).resize(im.size, Image.BILINEAR).save(MASKS / (p.stem + ".png"))
del teacher; torch.cuda.empty_cache()
print("masks drawn:", len(todo))

In [ ]:
# Side-by-side previews to check by eye. Delete the mask of every one that is wrong, then re-run section 5 on.
REVIEW = MINE / "review"; REVIEW.mkdir(exist_ok=True)
for p, m in pairs(IMAGES, MASKS):
    im = Image.open(p).convert("RGB"); im.thumbnail((512, 512))
    mk = Image.open(m).convert("L").resize(im.size)
    cut = Image.composite(im, Image.new("RGB", im.size, (255, 0, 255)), mk)
    both = Image.new("RGB", (im.width * 2, im.height)); both.paste(im, (0, 0)); both.paste(cut, (im.width, 0))
    both.save(REVIEW / (p.stem + ".jpg"), quality=85)
print("look through", REVIEW)

## 5. Data, and the number to beat

In [ ]:
SIDE = 320
MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

def app_input(im):
    """A picture exactly as LightDecoder gives it to the model (SubjectCutter.modelInput)."""
    a = np.asarray(im.convert("RGB").resize((SIDE, SIDE), Image.BILINEAR), np.float32)
    a = a / max(float(a.max()), 1.0)
    return (a.transpose(2, 0, 1) - MEAN) / STD

class Cuts(torch.utils.data.Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, m = self.items[i]
        im, mk = Image.open(p).convert("RGB"), Image.open(m).convert("L")
        if mk.size != im.size:
            mk = mk.resize(im.size, Image.BILINEAR)
        if self.train:
            if random.random() < 0.5:
                im, mk = ImageOps.mirror(im), ImageOps.mirror(mk)
            if random.random() < 0.7:           # a random part of the picture, so subjects vary in size and place
                w, h = im.size; s = random.uniform(0.7, 1.0)
                cw, ch = int(w * s), int(h * s); x, y = random.randint(0, w - cw), random.randint(0, h - ch)
                im, mk = im.crop((x, y, x + cw, y + ch)), mk.crop((x, y, x + cw, y + ch))
            if random.random() < 0.3:           # a quarter turn: phones hold pictures every way up
                k = random.choice([Image.ROTATE_90, Image.ROTATE_270])
                im, mk = im.transpose(k), mk.transpose(k)
            if random.random() < 0.5:           # light and colour
                from PIL import ImageEnhance
                im = ImageEnhance.Brightness(im).enhance(random.uniform(0.7, 1.3))
                im = ImageEnhance.Contrast(im).enhance(random.uniform(0.8, 1.2))
                im = ImageEnhance.Color(im).enhance(random.uniform(0.8, 1.2))
        x = torch.from_numpy(app_input(im).astype(np.float32))
        y = torch.from_numpy(np.asarray(mk.resize((SIDE, SIDE), Image.BILINEAR), np.float32)[None] / 255.0)
        return x, y

mine = pairs(IMAGES, MASKS)
random.Random(7).shuffle(mine); duts_s = duts[:]; random.Random(7).shuffle(duts_s)
hold_mine = mine[: max(1, len(mine) // 10)] if mine else []     # never trained on
train_mine = mine[len(hold_mine):]
hold_duts, train_duts = duts_s[:500], duts_s[500:]
print(f"train: {len(train_mine)} mine + {len(train_duts)} DUTS | held out: {len(hold_mine)} mine + {len(hold_duts)} DUTS")
if len(train_mine) < 200:
    print("WARNING: fewer than 200 of your own pictures. Expect little or no gain over the published model.")

In [ ]:
@torch.no_grad()
def measure(net, items, batch=16):
    """Mean error (lower is better) and overlap, IoU (higher is better), of the first output."""
    if not items:
        return float("nan"), float("nan")
    net.eval().to(DEVICE)
    mae = iou = n = 0
    for x, y in torch.utils.data.DataLoader(Cuts(items, False), batch_size=batch, num_workers=4):
        x, y = x.to(DEVICE), y.to(DEVICE)
        d = net(x)[0]
        mae += (d - y).abs().mean(dim=(1, 2, 3)).sum().item()
        a, b = d > 0.5, y > 0.5
        iou += ((a & b).sum(dim=(1, 2, 3)).float() / (a | b).sum(dim=(1, 2, 3)).float().clamp(min=1)).sum().item()
        n += len(x)
    return mae / n, iou / n

base = published()
before = {"mine": measure(base, hold_mine), "duts": measure(base, hold_duts)}
print("published model   mine: MAE %.4f IoU %.4f   DUTS: MAE %.4f IoU %.4f" % (*before["mine"], *before["duts"]))
del base

## 6. Fine-tune

- Starts from the published weights, low learning rate (it is already good; large steps would undo that).
- Every batch is half your pictures, half DUTS (`MINE_SHARE`), whatever their counts.
- Loss is U-2-Net's own: cross-entropy on all seven outputs.
- Saves the best model by held-out score after each epoch, so stopping early loses nothing.

If the GPU runs out of memory, lower `BATCH` to 4.

In [ ]:
EPOCHS, BATCH, LR, MINE_SHARE = 12, 8, 2e-5, 0.5
STEPS_PER_EPOCH = 1500

net = published().to(DEVICE)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=0.0)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * STEPS_PER_EPOCH, pct_start=0.05)
bce = torch.nn.BCELoss()

def loader(items, n):
    if not items or n == 0:
        return None
    ds = Cuts(items, True)
    sampler = torch.utils.data.RandomSampler(ds, replacement=True, num_samples=10**9)
    return iter(torch.utils.data.DataLoader(ds, batch_size=n, sampler=sampler, num_workers=4, drop_last=True, persistent_workers=True))

n_mine = int(round(BATCH * MINE_SHARE)) if train_mine else 0
it_mine, it_duts = loader(train_mine, n_mine), loader(train_duts, BATCH - n_mine)

def score(net):
    m, d = measure(net, hold_mine), measure(net, hold_duts)
    # Your pictures count; DUTS is there to catch forgetting.
    return (m[1] if hold_mine else d[1]), m, d

BEST = WORK / "u2net-finetuned.pth"
best = -1.0
from tqdm.auto import tqdm
for epoch in range(EPOCHS):
    net.train(); total = 0.0
    for step in tqdm(range(STEPS_PER_EPOCH), desc=f"epoch {epoch + 1}/{EPOCHS}"):
        parts = [next(it) for it in (it_mine, it_duts) if it is not None]
        x = torch.cat([p[0] for p in parts]).to(DEVICE); y = torch.cat([p[1] for p in parts]).to(DEVICE)
        outs = net(x)                                   # seven masks, already 0..1
        loss = sum(bce(o.clamp(1e-6, 1 - 1e-6), y) for o in outs)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step(); sched.step(); total += loss.item()
    s, m, d = score(net)
    print("epoch %d  loss %.4f   mine: MAE %.4f IoU %.4f   DUTS: MAE %.4f IoU %.4f" % (epoch + 1, total / STEPS_PER_EPOCH, *m, *d))
    if s > best:
        best = s; torch.save(net.state_dict(), BEST); print("  saved (best so far)")

## 7. Is it actually better?

In [ ]:
tuned = U2NET(3, 1); tuned.load_state_dict(torch.load(BEST, map_location="cpu"))
after = {"mine": measure(tuned, hold_mine), "duts": measure(tuned, hold_duts)}
for k in ("mine", "duts"):
    print("%-5s IoU %.4f -> %.4f    MAE %.4f -> %.4f" % (k, before[k][1], after[k][1], before[k][0], after[k][0]))

better_mine = (not hold_mine) or after["mine"][1] > before["mine"][1] + 0.005
kept_duts = after["duts"][1] > before["duts"][1] - 0.01
SHIP = bool(hold_mine) and better_mine and kept_duts
print("\nSHIP IT" if SHIP else "\nDO NOT SHIP: not better on your pictures, or it forgot general pictures. Add more checked pictures, or lower LR / MINE_SHARE.")

In [ ]:
# Look at it too: published | fine-tuned, on held-out pictures of yours. Numbers can hide ugly edges.
@torch.no_grad()
def cut(net, im):
    net.eval().to(DEVICE)
    d = net(torch.from_numpy(app_input(im).astype(np.float32))[None].to(DEVICE))[0][0, 0].cpu().numpy()
    d = (d - d.min()) / max(float(d.max() - d.min()), 1e-6)
    mk = Image.fromarray((d * 255).astype(np.uint8)).resize(im.size, Image.BILINEAR)
    return Image.composite(im, Image.new("RGB", im.size, (255, 0, 255)), mk)

COMPARE = WORK / "compare"; COMPARE.mkdir(exist_ok=True)
base = published()
for p, _ in (hold_mine or hold_duts)[:40]:
    im = Image.open(p).convert("RGB"); im.thumbnail((512, 512))
    row = Image.new("RGB", (im.width * 3, im.height))
    for i, piece in enumerate((im, cut(base, im), cut(tuned, im))):
        row.paste(piece, (i * im.width, 0))
    row.save(COMPARE / (p.stem + ".jpg"), quality=88)
print("picture | published | fine-tuned  ->", COMPARE)

## 8. Export for the app

One input `[1, 3, 320, 320]`, the mask as output 0, fixed size, in a single file. Then the same picture is run
through PyTorch and through ONNX Runtime (what the app uses) and the two must agree.

In [ ]:
import onnx, onnxruntime as ort

class FirstMask(torch.nn.Module):          # the app reads only output 0
    def __init__(self, net):
        super().__init__(); self.net = net
    def forward(self, x):
        return self.net(x)[0]

ONNX = WORK / "u2net.onnx"
export = FirstMask(tuned).cpu().eval()
dummy = torch.zeros(1, 3, SIDE, SIDE)
kw = dict(input_names=["input"], output_names=["mask"], opset_version=12, do_constant_folding=True)
try:
    torch.onnx.export(export, dummy, str(ONNX), dynamo=False, **kw)   # newer PyTorch: ask for the classic exporter
except TypeError:
    torch.onnx.export(export, dummy, str(ONNX), **kw)

model = onnx.load(str(ONNX)); onnx.checker.check_model(model)
assert not list(WORK.glob("u2net.onnx.data")), "the weights were written to a second file; the app needs one file"
dims = [d.dim_value for d in model.graph.input[0].type.tensor_type.shape.dim]
assert dims == [1, 3, SIDE, SIDE], dims

session = ort.InferenceSession(str(ONNX), providers=["CPUExecutionProvider"])
worst = 0.0
for p, _ in (hold_mine + hold_duts)[:20]:
    x = app_input(Image.open(p))[None].astype(np.float32)
    a = session.run(None, {session.get_inputs()[0].name: x})[0]
    with torch.no_grad():
        b = export(torch.from_numpy(x)).numpy()
    worst = max(worst, float(np.abs(a - b).max()))
print("largest difference between PyTorch and the app's runtime:", worst)
assert a.size == SIDE * SIDE and worst < 1e-3
print("size in bytes:", ONNX.stat().st_size)

## 9. Into the app

The cell below splits the file into the two parts the app ships (each under GitHub's 100 MB limit) and prints the one line to change.

Then, in the LightDecoder project:

1. Copy `u2net.onnx.0` and `u2net.onnx.1` over the ones in `app/src/main/assets/models/`.
2. In `app/src/main/java/com/lightdecoder/sticker/SubjectCutter.kt` set `FULL_OBJECTS_BYTES` to the number printed.
   (If the split makes more than two parts, also set `FULL_OBJECTS_PARTS`.) **If this number is wrong the app
   silently falls back to the small model**, so check it.
3. Add a line to `NOTICE` saying the weights were further trained by you, from U-2-Net (Apache-2.0), with masks from BiRefNet (MIT).
4. Bump the version, build, and try Smart cut on the pictures that used to go wrong.

In [ ]:
OUT = WORK / "for-the-app"; shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()
PART = 90 * 1024 * 1024
data = ONNX.read_bytes()
parts = [data[i:i + PART] for i in range(0, len(data), PART)]
for i, part in enumerate(parts):
    (OUT / f"u2net.onnx.{i}").write_bytes(part)
import hashlib
print("copy these into app/src/main/assets/models/:", [f.name for f in sorted(OUT.iterdir())])
print(f"private const val FULL_OBJECTS_PARTS = {len(parts)}")
print(f"private const val FULL_OBJECTS_BYTES = {len(data):_}L")
print("sha256:", hashlib.sha256(data).hexdigest())
if not SHIP:
    print("\nSection 7 said DO NOT SHIP. These files are no better than what the app already has.")

## If it does not come out better

- **Not better on your pictures:** too few, or masks not checked. Add pictures; re-check `mine/review/`.
- **Worse on DUTS (forgetting):** lower `LR` to `1e-5` or `MINE_SHARE` to `0.3`.
- **Edges rough but the shape right:** that is the 320x320 limit, not training. The next step would be a
  different model (for example IS-Net at 1024, about the same file size, several times slower on the phone),
  which needs app changes; this notebook does not cover it.
- **People still cut badly:** on Auto the app uses a different model for people (Robust Video Matting);
  this training does not touch it. Put people in your pictures only if you want the object model to handle them when chosen by hand.